In [3]:
# LOAD Packages 
import pandas as pd
import numpy as np
import awkward as ak
from collections import namedtuple
!pip install awkward_pandas
import matplotlib
from mpl_toolkits.mplot3d import Axes3D
import os
import torch
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import json
import h5py
import glob
import sys
from mpl_toolkits.axes_grid1 import make_axes_locatable
!pip install plotly
import plotly
import plotly.graph_objects as go

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
# Open edep-sim file: 
edep_sim_file = "/global/cfs/cdirs/dune/users/ehinkle/nd_prototypes_ana/sheep-model/sample_gen/test_electron.hdf5"
sim_h5 = h5py.File(edep_sim_file, 'r')

In [9]:
segments_event_data_dtype = np.dtype([
    ('dE', 'f8'),
    ('x', 'f8'),
    ('y', 'f8'),
    ('z', 'f8')])

class ShowerDataset(Dataset):
         
    def __init__(self, h5_file_dir, start_pos_range, detector_active_regions, np_random_seed=0):

        self._file_dir = h5_file_dir
        self._set_dataset_file_list()  # Get list of files in dataset directory
        self._set_events_per_file()  # Get number of events per file + file indices
        self._start_pos_range = start_pos_range
        self._detector_active_regions = detector_active_regions
        self._np_random_seed = np_random_seed

        # Set random seed for reproducibility
        np.random.seed(self._np_random_seed)


    def __len__(self):
        return np.sum(self._events_per_file)

    def __getitem__(self, idx):

        file_idx, event_idx = self._decode_idx(idx)  # Decode the global index into file and event indices
        h5_file_name = self._file_list[file_idx]
        h5_file = h5py.File(h5_file_name, 'r')  # Open the HDF5 file
        file_events = h5_file['events']  # Access the events dataset
        file_segments = h5_file['segments']
        event = file_events[event_idx]
        event_id = event['event_id']
        segments = file_segments[file_segments['event_id'] == event_id]
        true_KE_initial = np.sqrt(np.sum(np.square(event['pxyz_start'])))
        transformed_segments, segments_det_mask, start_pos, rotation_matrix = self._get_filtered_segments(segments, self._detector_active_regions)
        filtered_segments = transformed_segments[segments_det_mask]
        event_data = {
            'file': h5_file_name,
            'event_id': event_id,
            'true_KE_initial': true_KE_initial,
            'start_pos': start_pos,
            'rotation_matrix': rotation_matrix,
            'detector_active_regions': self._detector_active_regions,
            'filtered_segments': np.array(list(zip(filtered_segments['dE'], filtered_segments['x'], \
                                              filtered_segments['y'], filtered_segments['z'])), dtype=segments_event_data_dtype),
            'all_segments': np.array(list(zip(transformed_segments['dE'], transformed_segments['x'], \
                                              transformed_segments['y'], transformed_segments['z'])), dtype=segments_event_data_dtype)
        }
        return event_data

    # Method to get file_idx, event_idx pair from global idx
    def _decode_idx(self, idx):
        """Decode a global index into a file index and an event index."""
        file_idx = np.digitize(idx, self._event_total_by_file) - 1  # Find the file index
        #print("File index:", file_idx)  # Debugging line to check file index
        event_idx = idx - self._event_total_by_file[file_idx]  # Find the event index within that file
        return file_idx, event_idx

    # Method to convert random start position and start direction to set of filtered visible depositions
    def _get_filtered_segments(self, segments, det_bounds):
        ''' Method to convert random start position and start direction to set of filtered visible depositions
        Inputs:
            - start_pos: 3D vector of the start position of the shower (x, y, z)
            - rotation_matrix: 3x3 rotation matrix representing the desired start direction of the shower
            - segments: array of segments with dE, x, y, z
            - det_bounds: array of detector bounds for each detector module [[[xmin1, ymin1, zmin1], [xmax1, ymax1, zmax1]],
                                                                             [[xmin2, ymin2, zmin2], [xmax2, ymax2, zmax2]], ...] 
        Outputs:
            - transformed segments: array of segments with transformed positions
            - in_any_volume: boolean array indicating whether each segment is within any detector volume
            - start_pos: the sampled start position
            - rotation_matrix: the sampled rotation matrix
        '''
        start_pos = self._sample_random_start_position()  # Sample a random start position
        rotation_matrix = self._sample_random_rotation_matrix()  # Sample a random rotation matrix

        # Step 1: Transform segments to the new coordinate system (rotation + translation)
        segment_positions = np.array([segments['x'], segments['y'], segments['z']]).T # Shape (N, 3) where N is the number of segments
        #print("A few segment positions:", segment_positions[:5])  # Debugging line to check segment positions
        transformed_segments_xyz = segment_positions @ rotation_matrix.T + start_pos # Shape (N, 3) where N is the number of segments
        transformed_segments = np.array(list(zip(segments['dE'], transformed_segments_xyz[:, 0], transformed_segments_xyz[:, 1], transformed_segments_xyz[:, 2])), dtype=segments_event_data_dtype) # Shape (N, 4) where N is the number of segments

        # Step 2: Filter segments based on detector bounds
        # Separate min bounds and max bounds
        min_bounds = det_bounds[:, 0, :] # Shape (M, 3) where M is the number of detector modules/active volumes
        max_bounds = det_bounds[:, 1, :] # Shape (M, 3) where M is the number of detector modules/active volumes

        # Check if each segment is within min or max bounds for each dimension
        greq_mins = transformed_segments_xyz[:, None, :] >= min_bounds # Shape (N, M, 3)
        leq_maxs = transformed_segments_xyz[:, None, :] <= max_bounds # Shape (N, M, 3)

        # Check if xyz of segment is within min and max bounds for any volume
        within_bounds = np.all(greq_mins & leq_maxs, axis=2) # Shape (N, M) 
        in_any_volume = np.any(within_bounds, axis=1) # Shape (N,) 

        return transformed_segments, in_any_volume, start_pos, rotation_matrix
    
    # Method to get uniformly randomly sampled directions
    def _sample_random_rotation_matrix(self):
        """Generate a random rotation matrix"""

        # Step 1: Uniform spherical sampling (use z->theta to reduce oversampling at poles) 
        phi = np.random.uniform(0, 2 * np.pi)
        z = np.random.uniform(-1, 1)
        theta = np.arccos(z) 

        # Step 2: Get direction vector
        sin_theta = np.sin(theta)
        direction = np.array([
            sin_theta * np.cos(phi),
            sin_theta * np.sin(phi),
            z
        ])

        # Step 3: Get random "roll" angle (rotation around direction vector axis)
        psi = np.random.uniform(0, 2 * np.pi)

        # Step 4: Create orthonormal basis using direction vector and two additional vectors
        if abs(direction[0]) < 0.99:
            v = np.array([1, 0, 0])
        else:
            v = np.array([0, 1, 0])

        u1 = np.cross(v, direction)
        u1 /= np.linalg.norm(u1)
        u2 = np.cross(direction, u1)

        # Step 5: Rotate u1 and u2 basis vectors by roll angle (rotating around direction vector)
        b1 = np.cos(psi) * u1 + np.sin(psi) * u2
        b2 = -np.sin(psi) * u1 + np.cos(psi) * u2
        d = direction # not rotated because it is the axis of rotation

        # Step 6: Create rotation matrix
        rotation_matrix = np.column_stack((b1, b2, d)) # in SO(3)

        return rotation_matrix
    
    # Method to get random start position
    def _sample_random_start_position(self):
        """Sample a random start position within the given range"""
        x_start = np.random.uniform(self._start_pos_range[0][0], self._start_pos_range[1][0])
        y_start = np.random.uniform(self._start_pos_range[0][1], self._start_pos_range[1][1])
        z_start = np.random.uniform(self._start_pos_range[0][2], self._start_pos_range[1][2])
        return np.array([x_start, y_start, z_start])
    
    # Method to get list of files in dataset directory
    def _set_dataset_file_list(self):
        """Get list of files in dataset directory"""
        self._file_list = []

        for file in glob.glob(self._file_dir + '*.hdf5'):
            self._file_list.append(file)

        if len(self._file_list) == 0:
            raise ValueError("No files found in dataset directory: {}".format(self._file_dir)) 
        
    # Method to get number of events per file
    def _set_events_per_file(self):
        self._events_per_file = []
        for file_name in self._file_list:
            with h5py.File(file_name, 'r') as f:
                events = f['events']
                self._events_per_file.append(len(events))
        self._events_per_file = np.array(self._events_per_file)
        self._event_total_by_file = np.cumsum(self._events_per_file)  # Cumulative sum to get event indices
        self._event_total_by_file = np.insert(self._event_total_by_file, 0, 0)

In [12]:
# Load set of xyz/dE for shower
test_electron_dataset = ShowerDataset(
    h5_file_dir="/global/cfs/cdirs/dune/users/ehinkle/nd_prototypes_ana/sheep-model/sample_gen/SAMPLES/CONVERT2H5/",
    start_pos_range=np.array([[-60, -60, -60], [60, 60, 60]]), 
    detector_active_regions=np.array([[[  3.069, -62.076,   2.4620001],
                            [ 63.931,  62.076,  64.538]],
                           [[  3.069, -62.076, -64.538],
                            [ 63.931,  62.076,  -2.4620001]],
                           [[-63.931, -62.076,   2.462],
                            [ -3.069,  62.076,  64.537994]],
                           [[-63.931, -62.076, -64.538],
                            [ -3.069,  62.076, -2.4620001]]])
)

# Get the first event
event_data = test_electron_dataset[987]
print(f"Event ID: {event_data['event_id']}")
print(f"True Initial KE: {event_data['true_KE_initial']} MeV")
print(f"Number of segments: {len(event_data['filtered_segments']['dE'])}")   
print(f"Start position: {event_data['start_pos']}")
filtered_segments = event_data['filtered_segments']
all_segments = event_data['all_segments']
det_bounds_2x2 = event_data['detector_active_regions']

File index: 19
Event ID: 37
True Initial KE: 666.1806030273438 MeV
Number of segments: 13794
Start position: [ 5.85762047 25.82272396 12.33160513]
Event ID: 37
True Initial KE: 666.1806030273438 MeV
Number of segments: 13794
Start position: [ 5.85762047 25.82272396 12.33160513]


In [ ]:
# Plot original segments and filtered segments
def plot_segments(segments, filtered_segments):
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(segments['x'], segments['y'], segments['z'], color='blue',label="Original", marker='o', s=1)
    ax.scatter(filtered_segments['x'], filtered_segments['y'], filtered_segments['z'], color='red', label="Filtered", marker='o', s=4)
    ax.legend()
    ax.set_xlabel('X (cm)')
    ax.set_ylabel('Y (cm)')
    ax.set_zlabel('Z (cm)')
    ax.set_title("Filtered Segments from Random Start Position and Direction")
    plt.show()

# Plot original segments
#plot_segments(event_data['segments'], filtered_segments)

# Draw detector bounds
def draw_detector_bounds(ax, det_bounds):
    for bounds in det_bounds:
        # Draw outline of the detector bounds
        x_min, y_min, z_min = bounds[0]
        x_max, y_max, z_max = bounds[1]
        # Create a list of vertices for the bounding box
        vertices = np.array([[z_min, x_min, y_min],
                             [z_min, x_min, y_max],
                             [z_min, x_max, y_min],
                             [z_min, x_max, y_max],
                             [z_max, x_min, y_min],
                             [z_max, x_min, y_max],
                             [z_max, x_max, y_min],
                             [z_max, x_max, y_max]])
        # Create a list of edges connecting the vertices
        edges = [
            [vertices[0], vertices[1]], [vertices[0], vertices[2]],
            [vertices[0], vertices[4]], [vertices[1], vertices[3]]]
        edges += [[vertices[1], vertices[5]], [vertices[2], vertices[3]],
                  [vertices[2], vertices[6]], [vertices[3], vertices[7]]]
        edges += [[vertices[4], vertices[5]], [vertices[4], vertices[6]],
                  [vertices[5], vertices[7]], [vertices[6], vertices[7]]]

        # Plot the edges
        for edge in edges:
            ax.plot3D(*zip(*edge), color='green', linewidth=1)    

# Plot segments with detector bounds
def plot_segments_with_bounds(segments, filtered_segments, det_bounds):
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(segments['z'], segments['x'], segments['y'], color='blue', label="All", marker='o', s=1)
    ax.scatter(filtered_segments['z'], filtered_segments['x'], filtered_segments['y'], color='red', label="In Active Detector", marker='o', s=4)
    draw_detector_bounds(ax, det_bounds)
    ax.legend()
    ax.set_xlabel('Z (cm)')
    ax.set_ylabel('X (cm)')
    ax.set_zlabel('Y (cm)')
    ax.set_title("Filtered Segments with Detector Bounds")
    plt.show()

# Plot segments with detector bounds
plot_segments_with_bounds(all_segments, filtered_segments, det_bounds_2x2)

In [ ]:
import plotly.graph_objects as go
import numpy as np

def plot_segments_with_bounds_plotly(segments, filtered_segments, det_bounds):
    fig = go.Figure()

    # Original segments (blue)
    fig.add_trace(go.Scatter3d(
        x=segments['z'], y=segments['x'], z=segments['y'],
        mode='markers',
        marker=dict(size=1.5, color='blue'),
        name='Original'
    ))

    # Filtered segments (red)
    fig.add_trace(go.Scatter3d(
        x=filtered_segments['z'], y=filtered_segments['x'], z=filtered_segments['y'],
        mode='markers',
        marker=dict(size=3, color='red'),
        name='Filtered'
    ))

    # Detector bounds as wireframe boxes
    for bounds in det_bounds:
        x_min, y_min, z_min = bounds[0]
        x_max, y_max, z_max = bounds[1]

        # Vertices of the box
        x = [x_min, x_max]
        y = [y_min, y_max]
        z = [z_min, z_max]

        # 8 corners of the box
        corners = np.array([[z_min, x_min, y_min],
                            [z_min, x_min, y_max],
                            [z_min, x_max, y_min],
                            [z_min, x_max, y_max],
                            [z_max, x_min, y_min],
                            [z_max, x_min, y_max],
                            [z_max, x_max, y_min],
                            [z_max, x_max, y_max]])

        # List of 12 edges (pairs of corners)
        edge_indices = [
            (0,1), (0,2), (0,4), (1,3), (1,5), (2,3),
            (2,6), (3,7), (4,5), (4,6), (5,7), (6,7)
        ]

        for i, j in edge_indices:
            zi, xi, yi = corners[i]
            zj, xj, yj = corners[j]
            fig.add_trace(go.Scatter3d(
                x=[zi, zj], y=[xi, xj], z=[yi, yj],
                mode='lines',
                line=dict(color='green', width=2),
                showlegend=False
            ))

    # Layout settings
    fig.update_layout(
        scene=dict(
            xaxis_title='Z (cm)',
            yaxis_title='X (cm)',
            zaxis_title='Y (cm)',
            aspectmode='data'
        ),
        title="Filtered Segments with Detector Bounds (Plotly)",
        legend=dict(x=0.02, y=0.98),
        width=1000,  # Width in pixels
        height=800   # Height in pixels
    )

    fig.show()

plot_segments_with_bounds_plotly(all_segments, filtered_segments, det_bounds_2x2)

In [ ]:
# Demonstration of np.digitize
import numpy as np

# Example 1: Basic digitization
x = np.array([0.5, 1.2, 2.8, 3.5, 4.9])
bins = np.array([1, 2, 3, 4])

indices = np.digitize(x, bins)
print("Example 1: Basic digitization")
print(f"Values: {x}")
print(f"Bins: {bins}")
print(f"Bin indices: {indices}")
print()

# Example 2: Using with energy data (similar to your segments)
energies = np.array([0.1, 0.5, 1.2, 2.8, 5.1, 10.2])
energy_bins = np.array([0.5, 1.0, 2.0, 5.0, 10.0])  # Energy thresholds

energy_categories = np.digitize(energies, energy_bins)
print("Example 2: Energy categorization")
print(f"Energies (MeV): {energies}")
print(f"Energy bins: {energy_bins}")
print(f"Categories: {energy_categories}")
print()

# Show what each category means
category_names = ['Very Low', 'Low', 'Medium', 'High', 'Very High', 'Extreme']
for i, (energy, category) in enumerate(zip(energies, energy_categories)):
    print(f"Energy {energy} MeV → Category {category} ({category_names[category]})")